In [ ]:
# Cell 1: Import dependencies and resolve local config paths
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
import torch
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import to_categorical

notebook_dir = Path().resolve()
project_root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir

cfg_path = project_root / 'configs' / 'local_data_paths.yaml'
if cfg_path.exists():
    cfg = yaml.safe_load(cfg_path.read_text())
    dcfg = cfg.get('datasets', {}).get('noisy_drone_rf_v2', {})
    data_dir = Path(dcfg.get('data_dir', '/scratch/rameyjm7/datasets/NoisyDroneRFv2'))
else:
    data_dir = Path('/scratch/rameyjm7/datasets/NoisyDroneRFv2')

model_path = project_root / 'models' / 'noisy_drone_rf_v2' / 'noisy_drone_rf_v2_cnn_bilstm_final.keras'
print('Noisy Drone RF v2 data:', data_dir)
print('Noisy Drone RF v2 model:', model_path)
assert data_dir.exists(), f'Missing NoisyDroneRFv2 directory: {data_dir}'
assert model_path.exists(), f'Missing model: {model_path}'
outputs_dir = project_root / 'outputs' / 'noisy_drone_rf_v2_eval'
outputs_dir.mkdir(parents=True, exist_ok=True)
print('Outputs dir:', outputs_dir)


In [ ]:
# Cell 2: Load test split and build model input features
sample_re = re.compile(r'IQdata_sample(?P<sample>\d+)_target(?P<target>-?\d+)_snr(?P<snr>-?\d+)\.pt$')

def load_pt_iq(filepath):
    obj = torch.load(filepath, map_location='cpu')
    if isinstance(obj, dict):
        for key in ('iq', 'IQ', 'x', 'X', 'data', 'samples'):
            if key in obj:
                obj = obj[key]
                break
    elif isinstance(obj, (tuple, list)):
        obj = obj[0]

    arr = obj.detach().cpu().numpy() if hasattr(obj, 'detach') else np.asarray(obj)
    arr = np.asarray(arr, dtype=np.float32)
    arr = np.squeeze(arr)

    if arr.ndim == 1:
        if np.iscomplexobj(arr):
            arr = np.stack([arr.real, arr.imag], axis=-1).astype(np.float32)
        else:
            assert arr.size % 2 == 0, f'Cannot infer IQ pairs from shape {arr.shape} in {filepath}'
            arr = arr.reshape(-1, 2)
    elif arr.ndim == 2:
        if arr.shape[0] == 2 and arr.shape[1] != 2:
            arr = arr.T
        elif arr.shape[-1] != 2:
            raise ValueError(f'Expected IQ tensor with final dimension 2, got shape {arr.shape} in {filepath}')
    else:
        arr = arr.reshape(arr.shape[0], -1) if arr.shape[-1] != 2 else arr
        if arr.shape[-1] != 2 and arr.shape[0] == 2:
            arr = arr.T
        if arr.shape[-1] != 2:
            raise ValueError(f'Expected IQ tensor with final dimension 2, got shape {arr.shape} in {filepath}')

    return arr.astype(np.float32)

pt_files = sorted(data_dir.rglob('IQdata_sample*_target*_snr*.pt'))
assert pt_files, f'No IQdata_sample*_target*_snr*.pt files found under {data_dir}'

X_list, y_list, snr_list = [], [], []
for f in pt_files:
    m = sample_re.search(f.name)
    if not m:
        continue
    X_list.append(load_pt_iq(f))
    y_list.append(int(m.group('target')))
    snr_list.append(int(m.group('snr')))

min_len = min(x.shape[0] for x in X_list)
X = np.stack([x[:min_len, :2] for x in X_list], axis=0).astype(np.float32)
y_raw = np.asarray(y_list, dtype=np.int64)
snr = np.asarray(snr_list, dtype=np.float32)
classes = np.array(sorted(np.unique(y_raw)), dtype=np.int64)
class_to_index = {c: i for i, c in enumerate(classes)}
y = np.asarray([class_to_index[c] for c in y_raw], dtype=np.int64)
y_onehot = to_categorical(y, num_classes=len(classes))

idx = np.arange(X.shape[0])
train_idx, test_idx = train_test_split(idx, test_size=0.20, random_state=1961, stratify=y)

x_iq = X[test_idx]
y_true = y[test_idx]
snr_test = snr[test_idx]

def normalize_iq(X):
    Xn = np.empty_like(X)
    for i in range(X.shape[0]):
        scale = np.max(np.abs(X[i])) + 1e-12
        Xn[i] = X[i] / scale
    return Xn

x_iq = normalize_iq(x_iq)

def append_snr_feature(X, snr):
    out = []
    snr_scale = max(float(np.max(np.abs(snr))), 1.0)
    for i in range(X.shape[0]):
        snr_col = np.full((X.shape[1], 1), snr[i] / snr_scale)
        out.append(np.concatenate([X[i], snr_col], axis=1))
    return np.array(out, dtype=np.float32)

X_eval = append_snr_feature(x_iq, snr_test)

print('X_eval shape:', X_eval.shape)
print('class count:', len(classes))
print('snr_test available:', snr_test is not None)


In [ ]:
# Cell 3: Run model inference and print metrics
model = load_model(model_path, compile=False)
y_pred = np.argmax(model.predict(X_eval, verbose=0), axis=1)

acc = accuracy_score(y_true, y_pred)
print(f'Noisy Drone RF v2 evaluation accuracy: {acc:.4f}')
print(classification_report(y_true, y_pred, zero_division=0))


In [ ]:
# Cell 4: Plot confusion matrix for Noisy Drone RF v2
n_classes = int(max(y_true.max(), y_pred.max()) + 1)
cm = confusion_matrix(y_true, y_pred, labels=np.arange(n_classes))

plt.figure(figsize=(12, 10))
sns.heatmap(cm, cmap='Blues')
plt.title('Noisy Drone RF v2 Confusion Matrix (class index view)')
plt.xlabel('Predicted class index')
plt.ylabel('True class index')
plt.tight_layout()
plt.show()


In [ ]:
# Cell 5: Plot and save SNR line charts (overall and per class)
import csv

if snr_test is None:
    print('SNR metadata not available; skipping SNR line charts.')
else:
    snr_unique = np.array(sorted(np.unique(snr_test)), dtype=np.int64)
    n_classes = int(max(y_true.max(), y_pred.max()) + 1)

    overall_acc = []
    per_class = np.full((n_classes, len(snr_unique)), np.nan, dtype=np.float32)

    for j, snr_value in enumerate(snr_unique):
        idx = np.where(snr_test == snr_value)[0]
        overall_acc.append(float(np.mean(y_pred[idx] == y_true[idx])) * 100.0)

        for c in range(n_classes):
            m = idx[y_true[idx] == c]
            if len(m) > 0:
                per_class[c, j] = float(np.mean(y_pred[m] == y_true[m])) * 100.0

    fig, axes = plt.subplots(1, 2, figsize=(18, 7))

    axes[0].plot(snr_unique, overall_acc, marker='o', color='blue')
    axes[0].set_title('Recognition Accuracy vs. SNR (Noisy Drone RF v2)')
    axes[0].set_xlabel('SNR (dB)')
    axes[0].set_ylabel('Accuracy (%)')
    axes[0].grid(True, alpha=0.4)

    for c in range(n_classes):
        axes[1].plot(snr_unique, per_class[c], marker='o', linewidth=1.0, label=f'class_{c}')
    axes[1].set_title('Accuracy vs. SNR per Class (Noisy Drone RF v2)')
    axes[1].set_xlabel('SNR (dB)')
    axes[1].set_ylabel('Accuracy (%)')
    axes[1].grid(True, alpha=0.4)
    axes[1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=7)

    plt.tight_layout()
    png = outputs_dir / '44_noisy_drone_rf_v2_accuracy_vs_snr_line_plots.png'
    plt.savefig(png, dpi=180)
    print('Saved line charts:', png)
    plt.show()

    overall_csv = outputs_dir / '44_noisy_drone_rf_v2_accuracy_vs_snr_line.csv'
    with open(overall_csv, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['snr_db', 'accuracy_percent'])
        for s, a in zip(snr_unique, overall_acc):
            writer.writerow([int(s), f'{a:.6f}'])
    print('Saved overall line data:', overall_csv)
